In [9]:
import re
import numpy as np
from scipy.io import loadmat
from scipy.interpolate import griddata
import plotly.graph_objects as go
from datetime import datetime, timedelta

# ---------------------- USER PATHS (edit these) ----------------------
WW_PATH   = "I_fitting_WW_slices.mat"
EAST_PATH = "East_Felix_06_Dis.mat"
EQ_PATH   = "FA_Cl_ALL_simple.mat"
RIM_PATH  = "axial_calderaRim.m"
AMC_PATH  = "axial_amc_top_utm_norm_km_20000.xyz"
OUT_HTML  = "axial_3d_with_amc_slider.html"
# --------------------------------------------------------------------

def latlon2xy_no_rotate(lat, lon, lat0=45.9547, lon0=-130.0089, rotation_deg=0.0):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    xltkm = 111.19
    xlnkm = xltkm * np.cos(np.deg2rad(lat0))
    dlat_km = (lat - lat0) * xltkm
    dlon_km = (lon - lon0) * xlnkm
    theta = np.deg2rad(-rotation_deg)
    snr, csr = np.sin(theta), np.cos(theta)
    y = csr * dlat_km + snr * dlon_km
    x = csr * dlon_km - snr * dlat_km
    return x, y

def parse_caldera_rim_from_m(filepath):
    with open(filepath, "r") as f:
        text = f.read()
    m = re.search(r"calderaRim\s*=\s*\[(.*?)\];", text, flags=re.S)
    if not m:
        raise RuntimeError("Could not find calderaRim block in rim .m file.")
    block = m.group(1).strip()
    lons, lats = [], []
    for line in block.splitlines():
        parts = re.split(r"[,\s]+", line.strip())
        vals = []
        for p in parts:
            try:
                vals.append(float(p))
            except:
                pass
        if len(vals) >= 2:
            lons.append(vals[0])
            lats.append(vals[1])
    return np.array(lons, float), np.array(lats, float)

def try_extract_struct_array(mat, preferred_names=("Po_Clu","Felix")):
    keys = [k for k in mat.keys() if not k.startswith("__")]
    for nm in preferred_names:
        if nm in mat:
            return nm, np.atleast_1d(mat[nm])
    for k in keys:
        obj = np.atleast_1d(mat[k])
        if obj.dtype == object or hasattr(obj.flat[0], "__dict__"):
            return k, obj
    return None, None

def extract_field(arr, field):
    out = []
    for e in arr.flat:
        if hasattr(e, field):
            out.append(getattr(e, field))
        elif isinstance(e, np.void) and e.dtype.names and field in e.dtype.names:
            out.append(e[field])
        else:
            return None
    return np.array(out).squeeze()

def matlab_datenum_to_datetime(dn: float) -> datetime:
    days = int(np.floor(dn))
    frac = dn - days
    return datetime.fromordinal(days) + timedelta(days=frac) - timedelta(days=366)

# ---------------------- WEST wall ----------------------
ww = loadmat(WW_PATH, squeeze_me=True, struct_as_record=False)
fit_xyz = None
for k in ww.keys():
    if k.lower() == "fit_x_y_z":
        fit_xyz = np.asarray(ww[k], float)
        break
x1, y1, z1 = fit_xyz[:,0], fit_xyz[:,1], fit_xyz[:,2]
mask = (y1 >= -2.5) & (y1 <= 0.0)
x1, y1, z1 = x1[mask], y1[mask], z1[mask]
A = np.column_stack([x1**2, y1**2, x1*y1, x1, y1, np.ones_like(x1)])
a,b,c,d,e,f = np.linalg.lstsq(A, z1, rcond=None)[0]
g = 0.156
xg = np.linspace(np.nanmin(x1), np.nanmax(x1), 200)
yg = np.linspace(-3.0, 0.0, 200)
XgW, YgW = np.meshgrid(xg, yg)
ZgW = a*XgW**2 + b*YgW**2 + c*XgW*YgW + d*XgW + e*YgW + f + g
ZgW = np.where((ZgW < -1.6) | (ZgW > 0.0), np.nan, ZgW)

# ---------------------- EAST wall ----------------------
east = loadmat(EAST_PATH, squeeze_me=True, struct_as_record=False)
_, east_arr = try_extract_struct_array(east, preferred_names=("Felix",))
lat2 = extract_field(east_arr, "lat").astype(float)
lon2 = extract_field(east_arr, "lon").astype(float)
dep2 = extract_field(east_arr, "depth").astype(float)

lat_min, lat_max = 45.93, 45.97
lon_min, lon_max = -130.00, -129.975
em = (lat2>=lat_min) & (lat2<=lat_max) & (lon2>=lon_min) & (lon2<=lon_max) & (dep2<=2.5)
lat2, lon2, dep2 = lat2[em], lon2[em], dep2[em]
x2, y2 = latlon2xy_no_rotate(lat2, lon2)
z2 = -dep2

def poly33_design(y, z):
    return np.column_stack([np.ones_like(y), y, z, y**2, y*z, z**2, y**3, y**2*z, y*z**2, z**3])

coef = np.linalg.lstsq(poly33_design(y2, z2), x2, rcond=None)[0]
yg2 = np.linspace(np.nanmin(y2), np.nanmax(y2), 200)
zg2 = np.linspace(np.nanmin(z2), np.nanmax(z2), 200)
YgE, ZgE = np.meshgrid(yg2, zg2)
XgE = (poly33_design(YgE.ravel(), ZgE.ravel()) @ coef).reshape(YgE.shape)

P1 = np.array([1.57, -1.44, -0.01])
P2 = np.array([2.12, -2.74, -0.94])
dx, dy, dz = (P2 - P1)
t = (XgE - P1[0]) / dx
valid_t = (t>=0) & (t<=1) & (~np.isnan(t))
Y_expected = P1[1] + t*dy
Z_expected = P1[2] + t*dz
deltaY = 25.0
mask_line = valid_t & (np.abs(YgE - Y_expected) < deltaY) & (ZgE > Z_expected)
mask_custom = ((YgE < -2.5) & (ZgE > -0.8)) | (XgE < 0.75) | (XgE > 2.75) | mask_line | ((YgE < -2.2) & (ZgE > -0.5))
XgE = np.where(mask_custom, np.nan, XgE)
YgE = np.where(mask_custom, np.nan, YgE)
ZgE = np.where(mask_custom, np.nan, ZgE)

# ---------------------- AMC surface ----------------------
amc = np.loadtxt(AMC_PATH)[:, :3]
Xa, Ya, Za = amc[:,0], amc[:,1], amc[:,2]
xi = np.linspace(np.nanmin(Xa), np.nanmax(Xa), 180)
yi = np.linspace(np.nanmin(Ya), np.nanmax(Ya), 180)
XgA, YgA = np.meshgrid(xi, yi)
ZgA = griddata((Xa,Ya), Za, (XgA,YgA), method="linear")
Znear = griddata((Xa,Ya), Za, (XgA,YgA), method="nearest")
ZgA = np.where(np.isnan(ZgA), Znear, ZgA)

# ---------------------- Rim ----------------------
rim_lon, rim_lat = parse_caldera_rim_from_m(RIM_PATH)
RimX, RimY = latlon2xy_no_rotate(rim_lat, rim_lon)

# ---------------------- Earthquakes + time check ----------------------
fa = loadmat(EQ_PATH, squeeze_me=True, struct_as_record=False)
_, eq_arr = try_extract_struct_array(fa, preferred_names=("Po_Clu","Felix"))
lat = extract_field(eq_arr, "lat").astype(float)
lon = extract_field(eq_arr, "lon").astype(float)
dep = extract_field(eq_arr, "depth").astype(float)
on = extract_field(eq_arr, "on")  # may be None

Xeq, Yeq = latlon2xy_no_rotate(lat, lon)
Zeq = -dep

has_time = on is not None and np.all(np.isfinite(on)) and np.nanmax(on) > 1e5

# downsample for size (edit if you want more)
max_pts = 60000
if Xeq.size > max_pts:
    step = int(np.ceil(Xeq.size / max_pts))
    Xeq, Yeq, Zeq = Xeq[::step], Yeq[::step], Zeq[::step]
    if has_time:
        on = np.asarray(on, float)[::step]

fig = go.Figure()
fig.add_trace(go.Surface(x=XgW, y=YgW, z=ZgW, opacity=0.55, showscale=False, name="West wall"))
fig.add_trace(go.Surface(x=XgE, y=YgE, z=ZgE, opacity=0.55, showscale=False, name="East wall"))
fig.add_trace(go.Surface(x=XgA, y=YgA, z=ZgA, opacity=0.35, showscale=False, name="AMC surface"))
fig.add_trace(go.Scatter3d(x=RimX, y=RimY, z=np.zeros_like(RimX), mode="lines", line=dict(width=6), name="Caldera rim"))

base_n = len(fig.data)

if has_time:
    dts = np.array([matlab_datenum_to_datetime(float(v)) for v in np.asarray(on, float)])
    order = np.argsort(dts)
    dts, Xeq, Yeq, Zeq = dts[order], Xeq[order], Yeq[order], Zeq[order]
    month_labels = np.array([f"{dt.year:04d}-{dt.month:02d}" for dt in dts])
    unique_months = list(dict.fromkeys(month_labels.tolist()))

    # one trace per month; slider toggles cumulatively (no frames; much smaller)
    month_to_idx = {m: np.where(month_labels == m)[0] for m in unique_months}
    for m in unique_months:
        idx = month_to_idx[m]
        fig.add_trace(go.Scatter3d(
            x=Xeq[idx], y=Yeq[idx], z=Zeq[idx],
            mode="markers",
            marker=dict(size=2, color=Zeq[idx]),
            name=f"EQ {m}",
            visible=False
        ))

    # pick first month that actually has quakes
    first_i = next((i for i,m in enumerate(unique_months) if month_to_idx[m].size > 0), 0)

    # initial visibility: show surfaces + rim + quakes through first_i
    vis0 = [True]*base_n + [j <= first_i for j in range(len(unique_months))]
    for tr, vis in zip(fig.data, vis0):
        tr.visible = vis

    steps = []
    for i, m in enumerate(unique_months):
        vis = [True]*base_n + [j <= i for j in range(len(unique_months))]
        steps.append(dict(
            method="update",
            label=m,
            args=[{"visible": vis},
                  {"title": f"Axial Seamount — Rotatable 3D view (through {m})"}]
        ))
    fig.update_layout(sliders=[dict(active=first_i, pad={"t": 30}, currentvalue={"prefix": "Through: "}, steps=steps)])
else:
    fig.add_trace(go.Scatter3d(x=Xeq, y=Yeq, z=Zeq, mode="markers", marker=dict(size=2, color=Zeq), name="Earthquakes"))

fig.update_layout(
    title="Axial Seamount — Rotatable 3D view" + (" (time slider)" if has_time else " (no time field found)"),
    scene=dict(
        xaxis_title="X (km)",
        yaxis_title="Y (km)",
        zaxis_title="Depth (km)",
        aspectmode="data",
        zaxis=dict(range=[-2.2, 0.0]),
    ),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.update_scenes(camera=dict(eye=dict(x=1.8, y=1.8, z=0.8)))
fig.write_html(OUT_HTML, include_plotlyjs="cdn", full_html=True)
print(f"Wrote: {OUT_HTML} | has_time={has_time} | points={Xeq.size}")


Wrote: axial_3d_with_amc_slider.html | has_time=True | points=38600


In [13]:
import os
import re
import numpy as np
from scipy.io import loadmat
from scipy.interpolate import griddata
import plotly.graph_objects as go
from datetime import datetime, timedelta

# ---------------------- USER PATHS ----------------------
WW_PATH   = "I_fitting_WW_slices.mat"
EAST_PATH = "East_Felix_06_Dis.mat"
EQ_PATH   = "FA_Cl_ALL_simple.mat"
RIM_PATH  = "axial_calderaRim.m"
AMC_PATH  = "axial_amc_top_utm_norm_km_20000.xyz"
OUT_HTML  = "axial_3d_with_amc_slider_FULL.html"
# -------------------------------------------------------

# ---------------------- LOCAL FRAME (matches your working code) ----------------------
AXCC1_LAT0 = 45.9547
AXCC1_LON0 = -130.0089
ROT_DEG = 0.0

def latlon2xy_no_rotate(lat, lon, lat0=AXCC1_LAT0, lon0=AXCC1_LON0, rotation_deg=ROT_DEG):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    xltkm = 111.19
    xlnkm = xltkm * np.cos(np.deg2rad(lat0))
    dlat_km = (lat - lat0) * xltkm
    dlon_km = (lon - lon0) * xlnkm
    theta = np.deg2rad(-rotation_deg)
    snr, csr = np.sin(theta), np.cos(theta)
    y = csr * dlat_km + snr * dlon_km
    x = csr * dlon_km - snr * dlat_km
    return x, y

def parse_caldera_rim_from_m(filepath):
    with open(filepath, "r") as f:
        text = f.read()
    m = re.search(r"calderaRim\s*=\s*\[(.*?)\];", text, flags=re.S)
    if not m:
        raise RuntimeError("Could not find calderaRim block in rim .m file.")
    block = m.group(1).strip()
    lons, lats = [], []
    for line in block.splitlines():
        parts = re.split(r"[,\s]+", line.strip())
        vals = []
        for p in parts:
            try: vals.append(float(p))
            except: pass
        if len(vals) >= 2:
            lons.append(vals[0]); lats.append(vals[1])
    return np.array(lons, float), np.array(lats, float)

def matlab_datenum_to_datetime(dn: float) -> datetime:
    days = int(np.floor(dn))
    frac = dn - days
    return datetime.fromordinal(days) + timedelta(days=frac) - timedelta(days=366)

def must_exist(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}\n"
                                f"Run this script from the folder containing the files, "
                                f"or set {os.path.basename(path)} to an absolute path.")

# ---------------------- MAT helpers (robust) ----------------------
def find_struct_with_fields(mat, required=("lat","lon","depth")):
    """Find any variable in a .mat that looks like a MATLAB struct array with required fields."""
    keys = [k for k in mat.keys() if not k.startswith("__")]
    for k in keys:
        obj = np.atleast_1d(mat[k])
        try:
            e0 = obj.flat[0]
        except Exception:
            continue

        # object array (struct_as_record=False usually gives this)
        if hasattr(e0, "__dict__"):
            if all(hasattr(e0, f) for f in required):
                return k, obj

        # structured array
        if isinstance(e0, np.void) and e0.dtype.names:
            names = [n.lower() for n in e0.dtype.names]
            if all(r in names for r in required):
                return k, obj
    return None, None

def extract_field(arr, field):
    out = []
    for e in arr.flat:
        if hasattr(e, field):
            out.append(getattr(e, field))
        elif isinstance(e, np.void) and e.dtype.names and field in e.dtype.names:
            out.append(e[field])
        else:
            return None
    return np.array(out).squeeze()

# ---------------------- UTM -> lat/lon (WGS84) (ported from your MATLAB) ----------------------
def utm2ll_wgs84(E, N, zone, is_northern=True):
    a  = 6378137.0
    e  = 0.0818191908426215
    e2 = e**2
    ep2 = e2/(1-e2)
    k0 = 0.9996

    x = np.asarray(E, float) - 500000.0
    y = np.asarray(N, float)
    if not is_northern:
        y = y - 10000000.0

    lon0_deg = (zone - 1)*6 - 180 + 3
    lon0 = np.deg2rad(lon0_deg)

    M = y / k0
    mu = M / (a*(1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))

    e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    J1 = 3*e1/2 - 27*e1**3/32
    J2 = 21*e1**2/16 - 55*e1**4/32
    J3 = 151*e1**3/96
    J4 = 1097*e1**4/512

    fp = mu + J1*np.sin(2*mu) + J2*np.sin(4*mu) + J3*np.sin(6*mu) + J4*np.sin(8*mu)

    sinfp = np.sin(fp); cosfp = np.cos(fp); tanfp = np.tan(fp)
    C1 = ep2 * (cosfp**2)
    T1 = tanfp**2
    N1 = a / np.sqrt(1 - e2*(sinfp**2))
    R1 = a*(1 - e2) / (1 - e2*(sinfp**2))**(3/2)
    D = x / (N1*k0)

    Q1 = N1 * tanfp / R1
    Q2 = (D**2)/2
    Q3 = (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*ep2) * (D**4)/24
    Q4 = (61 + 90*T1 + 298*C1 + 45*T1**2 - 252*ep2 - 3*C1**2) * (D**6)/720
    lat = fp - Q1*(Q2 - Q3 + Q4)

    Q5 = D
    Q6 = (1 + 2*T1 + C1) * (D**3)/6
    Q7 = (5 - 2*C1 + 28*T1 - 3*C1**2 + 8*ep2 + 24*T1**2) * (D**5)/120
    lon = lon0 + (Q5 - Q6 + Q7) / cosfp

    return np.rad2deg(lat), np.rad2deg(lon)

# ---------------------- file sanity ----------------------
for p in [WW_PATH, EAST_PATH, EQ_PATH, RIM_PATH, AMC_PATH]:
    must_exist(p)

# ======================================================================
# 1) WEST WALL (same as your working method)
# ======================================================================
ww = loadmat(WW_PATH, squeeze_me=True, struct_as_record=False)
ww_key = next((k for k in ww.keys() if k.lower() == "fit_x_y_z"), None)
if ww_key is None:
    raise RuntimeError(f"Could not find fit_x_y_z in {WW_PATH}. Keys: {list(ww.keys())}")

fit_xyz = np.asarray(ww[ww_key], float)
x1, y1, z1 = fit_xyz[:,0], fit_xyz[:,1], fit_xyz[:,2]
m = (y1 >= -2.5) & (y1 <= 0.0)
x1, y1, z1 = x1[m], y1[m], z1[m]

A = np.column_stack([x1**2, y1**2, x1*y1, x1, y1, np.ones_like(x1)])
a,b,c,d,e,f = np.linalg.lstsq(A, z1, rcond=None)[0]
g = 0.156

xg = np.linspace(np.nanmin(x1), np.nanmax(x1), 200)
yg = np.linspace(-3.0, 0.0, 200)
XgW, YgW = np.meshgrid(xg, yg)
ZgW = a*XgW**2 + b*YgW**2 + c*XgW*YgW + d*XgW + e*YgW + f + g
ZgW = np.where((ZgW < -1.6) | (ZgW > 0.0), np.nan, ZgW)

# ======================================================================
# 2) EAST WALL (same as your working method)
# ======================================================================
east = loadmat(EAST_PATH, squeeze_me=True, struct_as_record=False)
_, east_arr = find_struct_with_fields(east, required=("lat","lon","depth"))
if east_arr is None:
    raise RuntimeError(f"Could not find a struct with lat/lon/depth in {EAST_PATH}. Keys: {list(east.keys())}")

lat2 = extract_field(east_arr, "lat").astype(float)
lon2 = extract_field(east_arr, "lon").astype(float)
dep2 = extract_field(east_arr, "depth").astype(float)

lat_min, lat_max = 45.93, 45.97
lon_min, lon_max = -130.00, -129.975
em = (lat2>=lat_min) & (lat2<=lat_max) & (lon2>=lon_min) & (lon2<=lon_max) & (dep2<=2.5)
lat2, lon2, dep2 = lat2[em], lon2[em], dep2[em]

x2, y2 = latlon2xy_no_rotate(lat2, lon2)
z2 = -dep2  # already negative depth

def poly33_design(y, z):
    return np.column_stack([np.ones_like(y), y, z, y**2, y*z, z**2, y**3, y**2*z, y*z**2, z**3])

coef = np.linalg.lstsq(poly33_design(y2, z2), x2, rcond=None)[0]
yg2 = np.linspace(np.nanmin(y2), np.nanmax(y2), 200)
zg2 = np.linspace(np.nanmin(z2), np.nanmax(z2), 200)
YgE, ZgE = np.meshgrid(yg2, zg2)
XgE = (poly33_design(YgE.ravel(), ZgE.ravel()) @ coef).reshape(YgE.shape)

P1 = np.array([1.57, -1.44, -0.01])
P2 = np.array([2.12, -2.74, -0.94])
dx, dy, dz = (P2 - P1)
t = (XgE - P1[0]) / dx
valid_t = (t>=0) & (t<=1) & (~np.isnan(t))
Y_expected = P1[1] + t*dy
Z_expected = P1[2] + t*dz
deltaY = 25.0
mask_line = valid_t & (np.abs(YgE - Y_expected) < deltaY) & (ZgE > Z_expected)
mask_custom = ((YgE < -2.5) & (ZgE > -0.8)) | (XgE < 0.75) | (XgE > 2.75) | mask_line | ((YgE < -2.2) & (ZgE > -0.5))
XgE = np.where(mask_custom, np.nan, XgE)
YgE = np.where(mask_custom, np.nan, YgE)
ZgE = np.where(mask_custom, np.nan, ZgE)

# ======================================================================
# 3) EARTHQUAKES (+ slider if time field exists)
# ======================================================================
fa = loadmat(EQ_PATH, squeeze_me=True, struct_as_record=False)
_, eq_arr = find_struct_with_fields(fa, required=("lat","lon","depth"))
if eq_arr is None:
    raise RuntimeError(f"Could not find a struct with lat/lon/depth in {EQ_PATH}. Keys: {list(fa.keys())}")

lat = extract_field(eq_arr, "lat").astype(float)
lon = extract_field(eq_arr, "lon").astype(float)
dep = extract_field(eq_arr, "depth").astype(float)
on  = extract_field(eq_arr, "on")  # optional

Xeq, Yeq = latlon2xy_no_rotate(lat, lon)
Zeq = -dep  # negative depth

has_time = on is not None and np.all(np.isfinite(on)) and np.nanmax(on) > 1e5

# downsample for performance
max_pts = 60000
if Xeq.size > max_pts:
    step = int(np.ceil(Xeq.size / max_pts))
    Xeq, Yeq, Zeq = Xeq[::step], Yeq[::step], Zeq[::step]
    if has_time:
        on = np.asarray(on, float)[::step]

# ======================================================================
# 4) CALDERA RIM
# ======================================================================
rim_lon, rim_lat = parse_caldera_rim_from_m(RIM_PATH)
RimX, RimY = latlon2xy_no_rotate(rim_lat, rim_lon)

# ======================================================================
# 5) AMC SURFACE (co-registered to same XY frame as quakes/walls/rim)
#    File columns: x_off(m), y_off(m), z_raw(m)
#    E = x_off + x_norm ; N = y_off + y_norm
#    z_plot_m = z_raw - 20000 + 1500
# ======================================================================
amc = np.loadtxt(AMC_PATH)[:, :3]
x_off_m, y_off_m, z_raw_m = amc[:,0], amc[:,1], amc[:,2]

x_norm = 419691.465854
y_norm = 5081627.38769
E = x_off_m + x_norm
N = y_off_m + y_norm

lat_amc, lon_amc = utm2ll_wgs84(E, N, zone=9, is_northern=True)
Xa, Ya = latlon2xy_no_rotate(lat_amc, lon_amc)

z_plot_m = z_raw_m - 20000 + 1500
Za = -z_plot_m / 1000.0  # km (sign depends on your convention)

# You asked: "depths should be negative"
# Earthquakes + walls are already negative. AMC often comes out positive -> flip AMC only.
Za = -Za

# grid AMC for a smooth Plotly surface
xi = np.linspace(np.nanmin(Xa), np.nanmax(Xa), 180)
yi = np.linspace(np.nanmin(Ya), np.nanmax(Ya), 180)
XgA, YgA = np.meshgrid(xi, yi)
ZgA = griddata((Xa,Ya), Za, (XgA,YgA), method="linear")
Znear = griddata((Xa,Ya), Za, (XgA,YgA), method="nearest")
ZgA = np.where(np.isnan(ZgA), Znear, ZgA)

# ======================================================================
# 6) PLOTLY FIGURE
#    - No vertical exaggeration: aspectmode="data"
#    - Auto z-range includes AMC so it won't vanish
# ======================================================================
fig = go.Figure()

fig.add_trace(go.Surface(x=XgW, y=YgW, z=ZgW, opacity=0.55, showscale=False, name="West wall"))
fig.add_trace(go.Surface(x=XgE, y=YgE, z=ZgE, opacity=0.55, showscale=False, name="East wall"))
fig.add_trace(go.Surface(x=XgA, y=YgA, z=ZgA, opacity=0.35, showscale=False, name="AMC surface"))
fig.add_trace(go.Scatter3d(x=RimX, y=RimY, z=np.zeros_like(RimX),
                           mode="lines", line=dict(width=6), name="Caldera rim"))

base_n = len(fig.data)

if has_time:
    dts = np.array([matlab_datenum_to_datetime(float(v)) for v in np.asarray(on, float)])
    order = np.argsort(dts)
    dts, Xeq, Yeq, Zeq = dts[order], Xeq[order], Yeq[order], Zeq[order]

    month_labels = np.array([f"{dt.year:04d}-{dt.month:02d}" for dt in dts])
    unique_months = list(dict.fromkeys(month_labels.tolist()))
    month_to_idx = {m: np.where(month_labels == m)[0] for m in unique_months}

    for m in unique_months:
        idx = month_to_idx[m]
        fig.add_trace(go.Scatter3d(
            x=Xeq[idx], y=Yeq[idx], z=Zeq[idx],
            mode="markers",
            marker=dict(size=2, color=Zeq[idx]),
            name=f"EQ {m}",
            visible=False
        ))

    first_i = next((i for i,m in enumerate(unique_months) if month_to_idx[m].size > 0), 0)
    vis0 = [True]*base_n + [j <= first_i for j in range(len(unique_months))]
    for tr, vis in zip(fig.data, vis0):
        tr.visible = vis

    steps = []
    for i, m in enumerate(unique_months):
        vis = [True]*base_n + [j <= i for j in range(len(unique_months))]
        steps.append(dict(
            method="update",
            label=m,
            args=[{"visible": vis},
                  {"title": f"Axial Seamount — Rotatable 3D view (through {m})"}]
        ))

    fig.update_layout(sliders=[dict(active=first_i, pad={"t": 30},
                                    currentvalue={"prefix": "Through: "},
                                    steps=steps)])
else:
    fig.add_trace(go.Scatter3d(
        x=Xeq, y=Yeq, z=Zeq,
        mode="markers",
        marker=dict(size=2, color=Zeq),
        name="Earthquakes"
    ))

# ---- Auto z-range so AMC + walls + quakes all show ----
zmin = np.nanmin([np.nanmin(ZgW), np.nanmin(ZgE), np.nanmin(ZgA), np.nanmin(Zeq)])
zmax = np.nanmax([np.nanmax(ZgW), np.nanmax(ZgE), np.nanmax(ZgA), np.nanmax(Zeq)])
zpad = 0.1

fig.update_layout(
    title="Axial Seamount — Rotatable 3D view" + (" (time slider)" if has_time else " (no time field found)"),
    scene=dict(
        xaxis_title="X (km)",
        yaxis_title="Y (km)",
        zaxis_title="Depth (km, negative)",
        aspectmode="data",  # TRUE scaling (kills vertical exaggeration)
        zaxis=dict(range=[zmin - zpad, zmax + zpad]),
    ),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.update_scenes(camera=dict(eye=dict(x=1.8, y=1.8, z=0.8)))

fig.write_html(OUT_HTML, include_plotlyjs="cdn", full_html=True)
print(f"Wrote: {OUT_HTML} | has_time={has_time} | EQ points={Xeq.size}")
print("AMC XY km:", float(np.nanmin(Xa)), float(np.nanmax(Xa)), float(np.nanmin(Ya)), float(np.nanmax(Ya)))
print("AMC Z km:", float(np.nanmin(Za)), float(np.nanmax(Za)))
print("Z range used:", float(zmin - zpad), float(zmax + zpad))


Wrote: axial_3d_with_amc_slider_FULL.html | has_time=True | EQ points=38600
AMC XY km: -2.1911306512491224 5.269933100092299 -7.831601383377074 6.768652177404066
AMC Z km: -2.7093999999999996 -1.0997000000000008
Z range used: -4.236999999999999 0.09996610183381291


In [26]:
import os
import re
import numpy as np
from scipy.io import loadmat
from scipy.interpolate import griddata
from scipy.spatial import ConvexHull
from matplotlib.path import Path
import plotly.graph_objects as go
from datetime import datetime, timedelta

# ---------------------- USER PATHS ----------------------
WW_PATH   = "I_fitting_WW_slices.mat"
EAST_PATH = "East_Felix_06_Dis.mat"
EQ_PATH   = "FA_Cl_ALL_simple.mat"
RIM_PATH  = "axial_calderaRim.m"
AMC_PATH  = "axial_amc_top_utm_norm_km_20000.xyz"
OUT_HTML  = "axial_3d_with_amc_slider_MASKED.html"
# -------------------------------------------------------

# ---------------------- LOCAL FRAME (matches your working code) ----------------------
AXCC1_LAT0 = 45.9547
AXCC1_LON0 = -130.0089
ROT_DEG = 0.0

def latlon2xy_no_rotate(lat, lon, lat0=AXCC1_LAT0, lon0=AXCC1_LON0, rotation_deg=ROT_DEG):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    xltkm = 111.19
    xlnkm = xltkm * np.cos(np.deg2rad(lat0))
    dlat_km = (lat - lat0) * xltkm
    dlon_km = (lon - lon0) * xlnkm
    theta = np.deg2rad(-rotation_deg)
    snr, csr = np.sin(theta), np.cos(theta)
    y = csr * dlat_km + snr * dlon_km
    x = csr * dlon_km - snr * dlat_km
    return x, y

def parse_caldera_rim_from_m(filepath):
    with open(filepath, "r") as f:
        text = f.read()
    m = re.search(r"calderaRim\s*=\s*\[(.*?)\];", text, flags=re.S)
    if not m:
        raise RuntimeError("Could not find calderaRim block in rim .m file.")
    block = m.group(1).strip()
    lons, lats = [], []
    for line in block.splitlines():
        parts = re.split(r"[,\s]+", line.strip())
        vals = []
        for p in parts:
            try: vals.append(float(p))
            except: pass
        if len(vals) >= 2:
            lons.append(vals[0]); lats.append(vals[1])
    return np.array(lons, float), np.array(lats, float)

def matlab_datenum_to_datetime(dn: float) -> datetime:
    days = int(np.floor(dn))
    frac = dn - days
    return datetime.fromordinal(days) + timedelta(days=frac) - timedelta(days=366)

def must_exist(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            f"Run this script from the folder containing the files, or set the path to an absolute path."
        )

# ---------------------- MAT helpers (robust) ----------------------
def find_struct_with_fields(mat, required=("lat","lon","depth")):
    keys = [k for k in mat.keys() if not k.startswith("__")]
    for k in keys:
        obj = np.atleast_1d(mat[k])
        try:
            e0 = obj.flat[0]
        except Exception:
            continue

        if hasattr(e0, "__dict__"):
            if all(hasattr(e0, f) for f in required):
                return k, obj

        if isinstance(e0, np.void) and e0.dtype.names:
            names = [n.lower() for n in e0.dtype.names]
            if all(r in names for r in required):
                return k, obj

    return None, None

def extract_field(arr, field):
    out = []
    for e in arr.flat:
        if hasattr(e, field):
            out.append(getattr(e, field))
        elif isinstance(e, np.void) and e.dtype.names and field in e.dtype.names:
            out.append(e[field])
        else:
            return None
    return np.array(out).squeeze()

# ---------------------- UTM -> lat/lon (WGS84) ----------------------
def utm2ll_wgs84(E, N, zone, is_northern=True):
    a  = 6378137.0
    e  = 0.0818191908426215
    e2 = e**2
    ep2 = e2/(1-e2)
    k0 = 0.9996

    x = np.asarray(E, float) - 500000.0
    y = np.asarray(N, float)
    if not is_northern:
        y = y - 10000000.0

    lon0_deg = (zone - 1)*6 - 180 + 3
    lon0 = np.deg2rad(lon0_deg)

    M = y / k0
    mu = M / (a*(1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))

    e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    J1 = 3*e1/2 - 27*e1**3/32
    J2 = 21*e1**2/16 - 55*e1**4/32
    J3 = 151*e1**3/96
    J4 = 1097*e1**4/512

    fp = mu + J1*np.sin(2*mu) + J2*np.sin(4*mu) + J3*np.sin(6*mu) + J4*np.sin(8*mu)

    sinfp = np.sin(fp); cosfp = np.cos(fp); tanfp = np.tan(fp)
    C1 = ep2 * (cosfp**2)
    T1 = tanfp**2
    N1 = a / np.sqrt(1 - e2*(sinfp**2))
    R1 = a*(1 - e2) / (1 - e2*(sinfp**2))**(3/2)
    D = x / (N1*k0)

    Q1 = N1 * tanfp / R1
    Q2 = (D**2)/2
    Q3 = (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*ep2) * (D**4)/24
    Q4 = (61 + 90*T1 + 298*C1 + 45*T1**2 - 252*ep2 - 3*C1**2) * (D**6)/720
    lat = fp - Q1*(Q2 - Q3 + Q4)

    Q5 = D
    Q6 = (1 + 2*T1 + C1) * (D**3)/6
    Q7 = (5 - 2*C1 + 28*T1 - 3*C1**2 + 8*ep2 + 24*T1**2) * (D**5)/120
    lon = lon0 + (Q5 - Q6 + Q7) / cosfp

    return np.rad2deg(lat), np.rad2deg(lon)

# ---------------------- simple 1D 2-means for AMC z_raw split ----------------------
def two_means_threshold(z):
    z = np.asarray(z, float)
    c1, c2 = np.percentile(z, [40, 60])
    for _ in range(60):
        d1 = np.abs(z - c1)
        d2 = np.abs(z - c2)
        m1 = d1 <= d2
        if m1.sum() == 0 or (~m1).sum() == 0:
            break
        nc1 = z[m1].mean()
        nc2 = z[~m1].mean()
        if abs(nc1 - c1) < 1e-8 and abs(nc2 - c2) < 1e-8:
            break
        c1, c2 = nc1, nc2
    thresh = 0.5 * (c1 + c2)
    return thresh, c1, c2

# ---------------------- file sanity ----------------------
for p in [WW_PATH, EAST_PATH, EQ_PATH, RIM_PATH, AMC_PATH]:
    must_exist(p)

# ======================================================================
# 1) WEST WALL
# ======================================================================
ww = loadmat(WW_PATH, squeeze_me=True, struct_as_record=False)
ww_key = next((k for k in ww.keys() if k.lower() == "fit_x_y_z"), None)
if ww_key is None:
    raise RuntimeError(f"Could not find fit_x_y_z in {WW_PATH}. Keys: {list(ww.keys())}")

fit_xyz = np.asarray(ww[ww_key], float)
x1, y1, z1 = fit_xyz[:, 0], fit_xyz[:, 1], fit_xyz[:, 2]
m = (y1 >= -2.5) & (y1 <= 0.0)
x1, y1, z1 = x1[m], y1[m], z1[m]

A = np.column_stack([x1**2, y1**2, x1*y1, x1, y1, np.ones_like(x1)])
a, b, c, d, e, f = np.linalg.lstsq(A, z1, rcond=None)[0]
g = 0.156

xg = np.linspace(np.nanmin(x1), np.nanmax(x1), 200)
yg = np.linspace(-3.0, 0.0, 200)
XgW, YgW = np.meshgrid(xg, yg)
ZgW = a*XgW**2 + b*YgW**2 + c*XgW*YgW + d*XgW + e*YgW + f + g
ZgW = np.where((ZgW < -1.6) | (ZgW > 0.0), np.nan, ZgW)

# ======================================================================
# 2) EAST WALL
# ======================================================================
east = loadmat(EAST_PATH, squeeze_me=True, struct_as_record=False)
_, east_arr = find_struct_with_fields(east, required=("lat", "lon", "depth"))
if east_arr is None:
    raise RuntimeError(f"Could not find a struct with lat/lon/depth in {EAST_PATH}. Keys: {list(east.keys())}")

lat2 = extract_field(east_arr, "lat").astype(float)
lon2 = extract_field(east_arr, "lon").astype(float)
dep2 = extract_field(east_arr, "depth").astype(float)

lat_min, lat_max = 45.93, 45.97
lon_min, lon_max = -130.00, -129.975
em = (lat2 >= lat_min) & (lat2 <= lat_max) & (lon2 >= lon_min) & (lon2 <= lon_max) & (dep2 <= 2.5)
lat2, lon2, dep2 = lat2[em], lon2[em], dep2[em]

x2, y2 = latlon2xy_no_rotate(lat2, lon2)
z2 = -dep2

def poly33_design(y, z):
    return np.column_stack([np.ones_like(y), y, z, y**2, y*z, z**2, y**3, y**2*z, y*z**2, z**3])

coef = np.linalg.lstsq(poly33_design(y2, z2), x2, rcond=None)[0]
yg2 = np.linspace(np.nanmin(y2), np.nanmax(y2), 200)
zg2 = np.linspace(np.nanmin(z2), np.nanmax(z2), 200)
YgE, ZgE = np.meshgrid(yg2, zg2)
XgE = (poly33_design(YgE.ravel(), ZgE.ravel()) @ coef).reshape(YgE.shape)

P1 = np.array([1.57, -1.44, -0.01])
P2 = np.array([2.12, -2.74, -0.94])
dx, dy, dz = (P2 - P1)
t = (XgE - P1[0]) / dx
valid_t = (t >= 0) & (t <= 1) & (~np.isnan(t))
Y_expected = P1[1] + t * dy
Z_expected = P1[2] + t * dz
deltaY = 25.0
mask_line = valid_t & (np.abs(YgE - Y_expected) < deltaY) & (ZgE > Z_expected)

mask_custom = (
    ((YgE < -2.5) & (ZgE > -0.8)) |
    (XgE < 0.75) | (XgE > 2.75) |
    mask_line |
    ((YgE < -2.2) & (ZgE > -0.5))
)
XgE = np.where(mask_custom, np.nan, XgE)
YgE = np.where(mask_custom, np.nan, YgE)
ZgE = np.where(mask_custom, np.nan, ZgE)

# ======================================================================
# 3) EARTHQUAKES (+ slider if time exists)
# ======================================================================
fa = loadmat(EQ_PATH, squeeze_me=True, struct_as_record=False)
_, eq_arr = find_struct_with_fields(fa, required=("lat", "lon", "depth"))
if eq_arr is None:
    raise RuntimeError(f"Could not find a struct with lat/lon/depth in {EQ_PATH}. Keys: {list(fa.keys())}")

lat = extract_field(eq_arr, "lat").astype(float)
lon = extract_field(eq_arr, "lon").astype(float)
dep = extract_field(eq_arr, "depth").astype(float)
on  = extract_field(eq_arr, "on")  # optional

Xeq, Yeq = latlon2xy_no_rotate(lat, lon)
Zeq = -dep

has_time = on is not None and np.all(np.isfinite(on)) and np.nanmax(on) > 1e5

# downsample for performance
max_pts = 60000
if Xeq.size > max_pts:
    step = int(np.ceil(Xeq.size / max_pts))
    Xeq, Yeq, Zeq = Xeq[::step], Yeq[::step], Zeq[::step]
    if has_time:
        on = np.asarray(on, float)[::step]

# ======================================================================
# 4) CALDERA RIM
# ======================================================================
rim_lon, rim_lat = parse_caldera_rim_from_m(RIM_PATH)
RimX, RimY = latlon2xy_no_rotate(rim_lat, rim_lon)

# ======================================================================
# 5) AMC SURFACE (mask to chamber footprint)
# ======================================================================
amc = np.loadtxt(AMC_PATH)[:, :3]
x_off_m, y_off_m, z_raw_m = amc[:, 0], amc[:, 1], amc[:, 2]

# UTM offsets -> absolute UTM
x_norm = 419691.465854
y_norm = 5081627.38769
E = x_off_m + x_norm
N = y_off_m + y_norm

# UTM -> lat/lon -> local km (same XY frame as quakes/walls/rim)
lat_amc, lon_amc = utm2ll_wgs84(E, N, zone=9, is_northern=True)
Xa, Ya = latlon2xy_no_rotate(lat_amc, lon_amc)

# Depth conversion from your MATLAB convention:
# z_plot_m = z_raw - 20000 + 1500  (typically negative)
z_plot_m = z_raw_m - 20000 + 1500
Za_km = (-z_plot_m / 1000.0)+0

# You want depths negative: earthquakes and walls are already negative.
# AMC often ends up opposite depending on file; enforce negative here:
Za_km = -Za_km

# ---- Auto-select "chamber" subset using 2-means on z_raw ----
thresh, c1, c2 = two_means_threshold(z_raw_m)
# Higher z_raw => shallower top (given z_plot_m = z_raw - const)
# Keep the "shallower" cluster = z_raw >= thresh
thresh = np.percentile(z_raw_m, 5)
keep = z_raw_m >= thresh

Xa_c, Ya_c, Za_c = Xa[keep], Ya[keep], Za_km[keep]

# ---- Build a footprint polygon (convex hull) in local km and mask outside it ----
hull = ConvexHull(np.column_stack([Xa_c, Ya_c]))
poly = np.column_stack([Xa_c[hull.vertices], Ya_c[hull.vertices]])
poly_path = Path(poly)

# Grid only the chamber subset, then mask outside hull
xi = np.linspace(np.nanmin(Xa_c), np.nanmax(Xa_c), 220)
yi = np.linspace(np.nanmin(Ya_c), np.nanmax(Ya_c), 220)
XgA, YgA = np.meshgrid(xi, yi)

ZgA = griddata((Xa_c, Ya_c), Za_c, (XgA, YgA), method="linear")
Znear = griddata((Xa_c, Ya_c), Za_c, (XgA, YgA), method="nearest")
ZgA = np.where(np.isnan(ZgA), Znear, ZgA)

# Apply polygon mask so it doesn't fill a rectangle
inside = poly_path.contains_points(np.column_stack([XgA.ravel(), YgA.ravel()])).reshape(XgA.shape)
ZgA = np.where(inside, ZgA, np.nan)

# ======================================================================
# 6) PLOTLY FIGURE + COLORS
# ======================================================================
fig = go.Figure()

# Fault surfaces: keep default look (no colorbars)
fig.add_trace(go.Surface(x=XgW, y=YgW, z=ZgW, opacity=0.55, showscale=False, name="West wall"))
fig.add_trace(go.Surface(x=XgE, y=YgE, z=ZgE, opacity=0.55, showscale=False, name="East wall"))

# AMC: green colorscale
amc_green = [
    [0.0, "rgb(0,40,0)"],
    [0.5, "rgb(0,140,0)"],
    [1.0, "rgb(120,255,120)"],
]
fig.add_trace(go.Surface(
    x=XgA, y=YgA, z=ZgA,
    opacity=0.40,
    colorscale=amc_green,
    showscale=False,
    name="AMC (masked)"
))

# Rim
fig.add_trace(go.Scatter3d(
    x=RimX, y=RimY, z=np.zeros_like(RimX),
    mode="lines",
    line=dict(width=6, color="black"),
    name="Caldera rim"
))

base_n = len(fig.data)

# Earthquakes: black -> blue colorscale
eq_colors = [[0.0, "black"], [1.0, "black"]]

if has_time:
    dts = np.array([matlab_datenum_to_datetime(float(v)) for v in np.asarray(on, float)])
    order = np.argsort(dts)
    dts, Xeq, Yeq, Zeq = dts[order], Xeq[order], Yeq[order], Zeq[order]

    month_labels = np.array([f"{dt.year:04d}-{dt.month:02d}" for dt in dts])
    unique_months = list(dict.fromkeys(month_labels.tolist()))
    month_to_idx = {m: np.where(month_labels == m)[0] for m in unique_months}

    for m in unique_months:
        idx = month_to_idx[m]
        fig.add_trace(go.Scatter3d(
            x=Xeq[idx], y=Yeq[idx], z=Zeq[idx],
            mode="markers",
            marker=dict(
                size=2,
                color=Zeq[idx],
                colorscale=eq_colors,
                cmin=np.nanmin(Zeq),
                cmax=np.nanmax(Zeq),
                showscale=False,  # keep only one scale if you want; turn on below if desired
            ),
            name=f"EQ {m}",
            visible=False
        ))

    first_i = next((i for i, m in enumerate(unique_months) if month_to_idx[m].size > 0), 0)

    vis0 = [True] * base_n + [j <= first_i for j in range(len(unique_months))]
    for tr, vis in zip(fig.data, vis0):
        tr.visible = vis

    steps = []
    for i, m in enumerate(unique_months):
        vis = [True] * base_n + [j <= i for j in range(len(unique_months))]
        steps.append(dict(
            method="update",
            label=m,
            args=[{"visible": vis},
                  {"title": f"Axial Seamount — Rotatable 3D view (through {m})"}]
        ))

    fig.update_layout(sliders=[dict(active=first_i, pad={"t": 30},
                                    currentvalue={"prefix": "Through: "},
                                    steps=steps)])
else:
    fig.add_trace(go.Scatter3d(
        x=Xeq, y=Yeq, z=Zeq,
        mode="markers",
        marker=dict(
            size=2,
            color=Zeq,
            colorscale=eq_colors,
            showscale=True,
            colorbar=dict(title="EQ depth (km)"),
        ),
        name="Earthquakes"
    ))

# ---- Auto z-range so AMC never disappears ----
zmin = np.nanmin([np.nanmin(ZgW), np.nanmin(ZgE), np.nanmin(ZgA), np.nanmin(Zeq)])
zmax = np.nanmax([np.nanmax(ZgW), np.nanmax(ZgE), np.nanmax(ZgA), np.nanmax(Zeq)])
zpad = 0.1

fig.update_layout(
    title="Axial Seamount — Rotatable 3D view" + (" (time slider)" if has_time else " (no time field found)"),
    scene=dict(
        xaxis_title="X (km)",
        yaxis_title="Y (km)",
        zaxis_title="Depth (km, negative)",
        aspectmode="data",  # physically correct scaling (no vertical exaggeration)
        zaxis=dict(range=[zmin - zpad, zmax + zpad]),
    ),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.update_scenes(camera=dict(eye=dict(x=1.8, y=1.8, z=0.8)))

fig.write_html(OUT_HTML, include_plotlyjs="cdn", full_html=True)
print(f"Wrote: {OUT_HTML} | has_time={has_time} | EQ points={Xeq.size}")
print(f"AMC split: thresh(z_raw)={thresh:.2f}, centers=({c1:.2f},{c2:.2f}), kept={keep.mean()*100:.1f}%")


Wrote: axial_3d_with_amc_slider_MASKED.html | has_time=True | EQ points=38600
AMC split: thresh(z_raw)=16276.60, centers=(16550.27,16993.20), kept=95.0%
